In [3]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [4]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-09-17 13:28:19--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M   121MB/s    in 0.2s    

2026-09-17 13:28:20 (121 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [5]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [6]:
# add your code here - consider creating a new cell for each section of code

user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

df_ratings_filtered = df_ratings[
    df_ratings['user'].isin(user_counts[user_counts >= 200].index) &
    df_ratings['isbn'].isin(book_counts[book_counts >= 100].index)
]

df = df_ratings_filtered.merge(
    df_books,
    on='isbn'
)

df = df.drop_duplicates(['user', 'title'])

book_user_matrix = df.pivot(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

book_user_matrix_sparse = csr_matrix(book_user_matrix.values)

model = NearestNeighbors(metric='cosine')

model.fit(book_user_matrix_sparse)

Ratings after filtering users: 527556
Ratings after filtering books: 13793
Final ratings: 13614
Merged dataframe shape: (13614, 5)
Book-user matrix shape: (99, 857)
Sparse matrix created successfully.
KNN model fitted successfully.


In [7]:
# function to return recommended books - this will be tested
def get_recommends(book=""):

    book_vector = book_user_matrix.loc[book].values.reshape(1, -1)

    distances, indices = model.kneighbors(
        book_vector,
        n_neighbors=6
    )

    recommended_books = []

    for i in range(5, 0, -1):
        recommended_books.append([
            book_user_matrix.index[indices[0][i]],
            distances[0][i]
        ])

    return [book, recommended_books]

In [8]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [['The Lovely Bones: A Novel', 0.7234864234924316], ["The Pilot's Wife : A Novel", 0.8192678689956665], ['The Joy Luck Club', 0.8198604583740234], ['The Notebook', 0.8236683011054993], ['Bel Canto: A Novel', 0.8247874975204468]]]
You haven't passed yet. Keep trying!
